# 96 — Extended Thinking
## Private chain-of-thought scratchpads with Anthropic's `budget_tokens` API
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/96-extended-thinking/extended_thinking_workbook.ipynb)

Most LLM responses are produced in a single forward pass — the model generates tokens left-to-right and the first plausible-sounding answer is what you get. **Extended thinking** gives the model a private scratchpad before it writes its reply. The scratchpad tokens are used for genuine reasoning (exploration, self-correction, backtracking) and are never surfaced to end users, but they meaningfully change the quality of the final answer.

This workshop uses `claude-3-7-sonnet-20250219` with `thinking={'type':'enabled','budget_tokens':8000}` to tackle three classic traps:
- **Modular math** — computing the last digit of 7^77 requires recognising a cyclical pattern
- **Wordplay** — a coin puzzle that snares anyone who reads too quickly
- **CRT** — the Cognitive Reflection Test bat-and-ball item that tricks ~80% of people on first read

For each puzzle we call the API twice — with and without thinking — and compare the answers and token costs.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — What extended thinking is and why it works |
| 2 | **Setup** — Install Anthropic SDK, configure API key |
| 3 | **Puzzles** — The three benchmark problems and why they expose shallow reasoning |
| 4 | **`ask()` function** — How to call the API with and without thinking |
| 5 | **Response anatomy** — ThinkingBlock vs TextBlock |
| 6 | **Head-to-head comparison** — Run all three puzzles, inspect results |
| 7 | **Token budget tradeoffs** — Cost vs quality curves |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier is fine)
- `ANTHROPIC_API_KEY` in `.env` or Colab Secrets (get one at console.anthropic.com)
- `anthropic` Python SDK ≥ 0.40

### Key References
> Anthropic, *Extended Thinking*, 2025 — https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking  
> Shane Frederick, *Cognitive Reflection and Decision Making*, Journal of Economic Perspectives 2005  
> Kahneman, *Thinking, Fast and Slow* — System 1 vs System 2 thinking (the theoretical motivation)

## Part 1 — Concepts: What Is Extended Thinking?

### The problem: fast pattern matching vs deliberate reasoning

Standard LLM inference is **System 1** in Kahneman's sense — fast, associative, and prone to plausible-sounding-but-wrong shortcuts. The bat-and-ball problem ($1.10 total, bat is $1.00 more than ball) triggers the intuitive answer of 10¢, which is wrong. Extended thinking adds a **System 2** layer — slow, deliberate, and self-correcting.

### How the API works

```
User prompt
    │
    ▼
┌──────────────────────────────────────────┐
│  THINKING BLOCK  (private, not returned) │
│  - Scratchpad for exploration            │
│  - Can try wrong paths and backtrack     │
│  - budget_tokens controls max spend      │
└──────────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────────┐
│  TEXT BLOCK  (returned to caller)        │
│  - Final polished answer                 │
│  - Informed by the thinking above        │
└──────────────────────────────────────────┘
```

### Key API constraint: `max_tokens` must cover BOTH blocks

```python
# WRONG — max_tokens only for text, thinking silently truncated
max_tokens = 1024

# CORRECT — budget for thinking + room for the answer
max_tokens = THINKING_BUDGET + 2048   # e.g. 8000 + 2048 = 10048
```

### When extended thinking helps (and when it doesn't)

| Task type | Thinking helps? | Why |
|-----------|-----------------|-----|
| Multi-step math | ✅ Strongly | Needs sequential calculation |
| Lateral puzzles / traps | ✅ Strongly | First intuition is often wrong |
| Code with complex logic | ✅ Moderately | Can reason through edge cases |
| Simple factual recall | ❌ Minimal | Extra tokens just cost money |
| Creative writing | ❌ Minimal | Fluency ≠ reasoning |
| Short classification | ❌ Minimal | Overhead outweighs gain |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "anthropic", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install. Run: pip install anthropic python-dotenv")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Loaded key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded key from .env file.")

key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-ant')}")

## Part 2 — Setup: Imports and Constants

The example uses the `anthropic` SDK directly — no LangGraph, no LangChain. This is intentional: extended thinking is a model-level feature that works at the raw API layer, and wrapping it in a framework would obscure what's happening.

Two key constants:
- **`MODEL`** — `claude-3-7-sonnet-20250219` is the first Claude model to support extended thinking
- **`THINKING_BUDGET`** — 8,000 tokens for the scratchpad. The model may use fewer; it's a ceiling, not a floor.

In [ ]:
import anthropic

MODEL = "claude-3-7-sonnet-20250219"
THINKING_BUDGET = 8000  # private scratchpad tokens; counts toward max_tokens

print(f"Model: {MODEL}")
print(f"Thinking budget: {THINKING_BUDGET:,} tokens")
print(f"Max tokens (with thinking): {THINKING_BUDGET + 2048:,} tokens")
print(f"Max tokens (without thinking): 1024 tokens")

## Part 3 — The Three Benchmark Puzzles

These three problems are carefully chosen to expose different failure modes of shallow reasoning.

### Puzzle 1: Modular Math — Last digit of 7^77

The naive approach (compute 7^77 directly) fails for large exponents. The correct approach requires recognising the **cyclical pattern** in the last digits of powers of 7:

| Exponent | Value | Last digit |
|----------|-------|------------|
| 7^1 | 7 | **7** |
| 7^2 | 49 | **9** |
| 7^3 | 343 | **3** |
| 7^4 | 2401 | **1** |
| 7^5 | 16807 | **7** ← cycle repeats |

The cycle length is 4. `77 mod 4 = 1`, so 7^77 has the same last digit as 7^1 = **7**.

### Puzzle 2: Wordplay — The coin trap

> "I have exactly two coins that add up to 30 cents. One of them is not a nickel."

The trap: readers hear "one of them is not a nickel" and conclude *neither* is a nickel. But the statement only says **one** isn't. The answer is a **quarter + nickel** (25¢ + 5¢ = 30¢). The nickel is there; only the quarter "is not a nickel".

### Puzzle 3: CRT — Bat and ball

> "A bat and a ball together cost $1.10. The bat costs exactly $1.00 more than the ball."

The intuitive System-1 answer is 10¢. The correct answer is **5¢**:
```
Let ball = x cents
bat = x + 100 cents
(x + 100) + x = 110
2x = 10
x = 5
```
If ball = 10¢, then bat = $1.10, total = $1.20 — wrong.

In [ ]:
# Three problems chosen to expose where private reasoning changes the answer.
# CRT and Wordplay catch naive pattern-matching. Modular math needs real work.
PUZZLES = [
    {
        "label": "Modular Math",
        "prompt": "What is the last digit of 7^77? Show your reasoning step by step.",
    },
    {
        "label": "Wordplay",
        "prompt": (
            "I have exactly two coins that add up to 30 cents. "
            "One of them is not a nickel. What are the two coins?"
        ),
    },
    {
        "label": "CRT (Cognitive Reflection)",
        "prompt": (
            "A bat and a ball together cost $1.10. "
            "The bat costs exactly $1.00 more than the ball. "
            "How much does the ball cost? Give your answer in cents."
        ),
    },
]

print(f"Loaded {len(PUZZLES)} puzzles.")
for p in PUZZLES:
    print(f"  [{p['label']}] {p['prompt'][:60]}...")

## Part 4 — The `ask()` Function

The core of this example is a single function that calls Claude with or without extended thinking. Let's build it up piece by piece before running it.

### Request structure WITHOUT thinking

```python
{
    "model": "claude-3-7-sonnet-20250219",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 1024
    # No 'thinking' key
}
```

### Request structure WITH thinking

```python
{
    "model": "claude-3-7-sonnet-20250219",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 10048,          # MUST be >= budget_tokens
    "thinking": {
        "type": "enabled",
        "budget_tokens": 8000
    }
}
```

### Parsing the response

The response `content` list contains either:
- `[TextBlock]` — without thinking (one block)
- `[ThinkingBlock, TextBlock]` — with thinking (two blocks)

We iterate over `response.content` and dispatch by `block.type`.

In [ ]:
def ask(prompt: str, thinking: bool) -> dict:
    """Call Claude with or without the extended thinking scratchpad.

    With thinking enabled the response contains two ContentBlocks:
      - ThinkingBlock  (type='thinking')  -- private chain-of-thought, not shown to users
      - TextBlock      (type='text')      -- the final answer returned to the caller
    max_tokens must cover both thinking and text, so we size it generously.
    """
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    kwargs: dict = {
        "model":    MODEL,
        "messages": [{"role": "user", "content": prompt}],
        # When thinking is on, max_tokens must be >= budget_tokens.
        "max_tokens": THINKING_BUDGET + 2048 if thinking else 1024,
    }
    if thinking:
        kwargs["thinking"] = {"type": "enabled", "budget_tokens": THINKING_BUDGET}

    response = client.messages.create(**kwargs)

    thinking_text = ""
    answer = ""
    for block in response.content:
        if block.type == "thinking":
            thinking_text = block.thinking
        elif block.type == "text":
            answer = block.text

    return {
        "answer":           answer.strip(),
        "thinking_preview": thinking_text[:300] if thinking_text else "",
        "thinking_words":   len(thinking_text.split()) if thinking_text else 0,
        "output_tokens":    response.usage.output_tokens,
    }

print("ask() function defined.")

## Part 5 — Response Anatomy: ThinkingBlock vs TextBlock

Before running all three puzzles, let's run a single lightweight call to inspect what a raw response looks like with thinking enabled.

This cell runs a simple arithmetic question. Watch:
1. How many blocks are in `response.content`
2. The `type` attribute of each block
3. The `usage.output_tokens` count (thinking tokens are included here)

> **Note:** The `ThinkingBlock.thinking` field is the raw scratchpad text. Anthropic's API policy is that this is genuinely private — it should never be displayed to end users in production applications.

In [ ]:
# Inspect a raw API response to see the two-block structure
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

test_prompt = "What is 17 * 23? Show your work."

response = client.messages.create(
    model=MODEL,
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=THINKING_BUDGET + 2048,
    thinking={"type": "enabled", "budget_tokens": THINKING_BUDGET},
)

print(f"Number of content blocks: {len(response.content)}")
print(f"Total output tokens used: {response.usage.output_tokens:,}")
print()

for i, block in enumerate(response.content):
    print(f"Block {i}: type='{block.type}'")
    if block.type == "thinking":
        words = len(block.thinking.split())
        print(f"  Thinking words: {words}")
        print(f"  First 200 chars: {block.thinking[:200]}")
    elif block.type == "text":
        print(f"  Answer: {block.text[:200]}")
    print()

## Part 6 — Head-to-Head Comparison: All Three Puzzles

Now we run the actual benchmark: each puzzle twice — once with thinking disabled, once with thinking enabled. We compare:
- The **answer** (is it correct?)
- The **token cost** (thinking is expensive)
- The **thinking word count** (proxy for how hard the model worked)

> This will make 6 API calls total. Cost estimate: ~$0.05–$0.15 depending on thinking depth.

In [ ]:
print("=== 96 · Extended Thinking ===")
print(f"Model: {MODEL}")
print(f"Thinking budget: {THINKING_BUDGET:,} tokens")
print("Thinking blocks are private — they improve quality but are never returned to users.\n")
print("-" * 70)

In [ ]:
# Run the first puzzle: Modular Math
puzzle = PUZZLES[0]
label  = puzzle["label"]
prompt = puzzle["prompt"]

print(f"[{label}]")
print(f"Q: {prompt}\n")

plain   = ask(prompt, thinking=False)
thought = ask(prompt, thinking=True)

print(f"  Without thinking ({plain['output_tokens']} tokens):")
print(f"    {plain['answer'][:300]}")
print()
print(f"  With thinking ({thought['output_tokens']} tokens | ~{thought['thinking_words']} thinking words):")
print(f"    {thought['answer'][:300]}")
print()
print(f"  Thinking preview: {thought['thinking_preview']}...")

In [ ]:
# Run the second puzzle: Wordplay (coin trap)
puzzle = PUZZLES[1]
label  = puzzle["label"]
prompt = puzzle["prompt"]

print(f"[{label}]")
print(f"Q: {prompt}\n")

plain   = ask(prompt, thinking=False)
thought = ask(prompt, thinking=True)

print(f"  Without thinking ({plain['output_tokens']} tokens):")
print(f"    {plain['answer'][:300]}")
print()
print(f"  With thinking ({thought['output_tokens']} tokens | ~{thought['thinking_words']} thinking words):")
print(f"    {thought['answer'][:300]}")
print()
print(f"  Thinking preview: {thought['thinking_preview']}...")

In [ ]:
# Run the third puzzle: CRT (Cognitive Reflection Test)
puzzle = PUZZLES[2]
label  = puzzle["label"]
prompt = puzzle["prompt"]

print(f"[{label}]")
print(f"Q: {prompt}\n")

plain   = ask(prompt, thinking=False)
thought = ask(prompt, thinking=True)

print(f"  Without thinking ({plain['output_tokens']} tokens):")
print(f"    {plain['answer'][:300]}")
print()
print(f"  With thinking ({thought['output_tokens']} tokens | ~{thought['thinking_words']} thinking words):")
print(f"    {thought['answer'][:300]}")
print()
print(f"  Thinking preview: {thought['thinking_preview']}...")

## Part 6b — Full Benchmark Loop (Compact Output)

This cell runs all three puzzles in the same loop as the original `main.py`, printing a compact side-by-side comparison. Run it only if you want to re-run all three at once.

In [ ]:
# Full loop matching main.py output format exactly
results = []

for puzzle in PUZZLES:
    label  = puzzle["label"]
    prompt = puzzle["prompt"]
    print(f"[{label}]")
    print(f"Q: {prompt}\n")

    plain   = ask(prompt, thinking=False)
    thought = ask(prompt, thinking=True)

    results.append({"label": label, "plain": plain, "thought": thought})

    print(f"  Without thinking ({plain['output_tokens']} tokens):")
    print(f"    {plain['answer'][:200]}")
    print()
    print(f"  With thinking ({thought['output_tokens']} tokens | ~{thought['thinking_words']} thinking words):")
    print(f"    {thought['answer'][:200]}")
    print()
    print(f"  Thinking preview: {thought['thinking_preview']}...")
    print("-" * 70)
    print()

print("Takeaway: extended thinking helps most on problems where the first")
print("intuitive answer is wrong (CRT) or requires step-by-step calculation.")

## Part 7 — Token Budget Tradeoffs

### Cost breakdown

Extended thinking tokens are billed at the same rate as regular output tokens. The `budget_tokens` parameter is a **ceiling** — the model decides how many thinking tokens to actually use based on problem difficulty.

### Budget strategy

| Budget | Good for | Rough cost per call |
|--------|----------|---------------------|
| 1,000  | Simple logic, light verification | Very cheap |
| 5,000  | Multi-step math, mild puzzles | Moderate |
| 8,000  | Hard CRT-style problems, complex code | Moderate-high |
| 16,000 | Research-grade reasoning, proofs | High |
| 32,000 | Maximum (model cap) | Very high |

### The `output_tokens` signal

`response.usage.output_tokens` includes **both** thinking tokens and text tokens. If this number is close to `max_tokens`, you're hitting the ceiling — increase `max_tokens` or reduce `budget_tokens`.

### Practical rules
1. Start with `budget_tokens=5000` and increase until quality plateaus
2. Always set `max_tokens >= budget_tokens + expected_answer_length`
3. For real-time applications, think about latency — 8,000 thinking tokens adds noticeable delay
4. Cache thinking-heavy responses when the same question will be asked repeatedly

In [ ]:
# Compare different thinking budgets on the CRT puzzle (most sensitive to budget size)
crt_prompt = PUZZLES[2]["prompt"]

budgets = [0, 1000, 4000, 8000]

print(f"Q: {crt_prompt}")
print("=" * 70)

for budget in budgets:
    if budget == 0:
        result = ask(crt_prompt, thinking=False)
        label = "No thinking"
        words = 0
    else:
        client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        response = client.messages.create(
            model=MODEL,
            messages=[{"role": "user", "content": crt_prompt}],
            max_tokens=budget + 2048,
            thinking={"type": "enabled", "budget_tokens": budget},
        )
        thinking_text = ""
        answer = ""
        for block in response.content:
            if block.type == "thinking":
                thinking_text = block.thinking
            elif block.type == "text":
                answer = block.text
        result = {
            "answer": answer.strip(),
            "output_tokens": response.usage.output_tokens,
        }
        words = len(thinking_text.split())
        label = f"budget={budget:,}"

    print(f"\n[{label}] ({result['output_tokens']} output tokens, ~{words} thinking words)")
    print(f"  Answer: {result['answer'][:200]}")

## Part 8 — When Thinking Is Private (The Production Rule)

A critical point from the Anthropic documentation and the code's own comments:

> *"Thinking blocks are private — they improve quality but are never returned to users."*

### What this means in practice

In `ask()`, the `thinking_preview` key is included in the **returned dict**, but this is for **developer inspection only** — it should never flow into a user-facing response. The `ask()` function correctly separates:
- `thinking_preview` — developer tool (first 300 chars of scratchpad)
- `answer` — what gets sent to the user

### Why Anthropic enforces this

The thinking block may contain:
- Exploratory reasoning that was later corrected
- Intermediate wrong answers
- Uncertain hedging that would confuse users
- Potentially sensitive information if the model reasons about context

The final `TextBlock` is the model's **committed, polished answer** after all that internal deliberation.

In [ ]:
# Demonstrate the separation of concerns in ask()
result = ask(PUZZLES[0]["prompt"], thinking=True)

print("=== What the USER sees ===")
print(result["answer"])

print()
print("=== What the DEVELOPER can inspect (NEVER show to users) ===")
print(f"Thinking word count: {result['thinking_words']}")
print(f"Thinking preview (first 300 chars): {result['thinking_preview']}")

print()
print("=== Token accounting ===")
print(f"Total output tokens (thinking + text): {result['output_tokens']}")

## Part 9 — Building a Custom Puzzle Runner

Now let's write a more general version of the runner that can handle any prompt and report a structured comparison. This is the pattern you'd use in a production evaluation harness.

In [ ]:
def compare_thinking(
    prompt: str,
    budget: int = THINKING_BUDGET,
    answer_preview_chars: int = 300
) -> dict:
    """Run prompt with and without thinking, return structured comparison."""
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    # Without thinking
    r_plain = client.messages.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
    )
    plain_answer = r_plain.content[0].text.strip()
    plain_tokens = r_plain.usage.output_tokens

    # With thinking
    r_thought = client.messages.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=budget + 2048,
        thinking={"type": "enabled", "budget_tokens": budget},
    )
    thinking_text = ""
    thought_answer = ""
    for block in r_thought.content:
        if block.type == "thinking":
            thinking_text = block.thinking
        elif block.type == "text":
            thought_answer = block.text.strip()
    thought_tokens = r_thought.usage.output_tokens

    return {
        "prompt": prompt,
        "plain": {
            "answer": plain_answer[:answer_preview_chars],
            "tokens": plain_tokens,
        },
        "thought": {
            "answer": thought_answer[:answer_preview_chars],
            "tokens": thought_tokens,
            "thinking_words": len(thinking_text.split()),
            "thinking_preview": thinking_text[:300],
        },
        "token_overhead": thought_tokens - plain_tokens,
    }

print("compare_thinking() defined.")

In [ ]:
# Test compare_thinking() on a custom prompt
custom_prompt = "If I fold a piece of paper in half 10 times, how many layers does it have?"

result = compare_thinking(custom_prompt, budget=4000)

print(f"Q: {result['prompt']}")
print()
print(f"WITHOUT thinking ({result['plain']['tokens']} tokens):")
print(f"  {result['plain']['answer']}")
print()
print(f"WITH thinking ({result['thought']['tokens']} tokens, ~{result['thought']['thinking_words']} thinking words):")
print(f"  {result['thought']['answer']}")
print()
print(f"Token overhead from thinking: +{result['token_overhead']}")

## Part 10 — Summary and Key Takeaways

### What we built
- A direct `anthropic` SDK integration calling `claude-3-7-sonnet-20250219`
- An `ask()` function that toggles extended thinking via the `thinking` parameter
- A three-puzzle benchmark covering modular math, wordplay, and CRT
- A `compare_thinking()` utility for any custom prompt

### Key architectural insights

1. **`max_tokens` must cover both blocks.** Set `max_tokens >= budget_tokens + expected_text_length`.
2. **ThinkingBlock is always private.** Never expose `block.thinking` to end users.
3. **Budget is a ceiling, not a floor.** The model uses as many thinking tokens as needed for the problem.
4. **Token overhead is real.** A 8,000-token budget call costs ~8x–10x more than a no-thinking call.
5. **Quality gains are task-specific.** CRT-style traps and multi-step math benefit most; simple retrieval gains little.

### Where to go next
- Combine extended thinking with **tool use** (thinking block can reason about which tool to call)
- Build an **eval harness** that measures accuracy on a benchmark dataset with and without thinking
- Experiment with **streaming** the response — thinking blocks stream separately from text blocks
- Try **prompt engineering** to guide the thinking block (e.g., "Think about this step by step...")

---
## ★ Exercises

### Exercise 1 — Add a new puzzle

Add a fourth puzzle to `PUZZLES` and run it through both modes. Choose a problem where you expect extended thinking to help (e.g., a probability puzzle or a logic problem with a counterintuitive answer). Compare the results.

**Starter ideas:**
- The Monty Hall problem ("You pick door 1. The host opens door 3 revealing a goat. Should you switch?")
- The birthday paradox ("How many people do you need in a room for a 50%+ chance two share a birthday?")
- A simple syllogism trap ("All roses are flowers. Some flowers fade quickly. Do some roses fade quickly?")

### Exercise 2 — Budget sweep

Write a function `budget_sweep(prompt, budgets)` that runs `compare_thinking` for a list of budget values (e.g., `[500, 2000, 5000, 10000]`) and returns a table of `(budget, answer, tokens)`. Run it on the CRT puzzle. At what budget does the answer stabilise?

### Exercise 3 — Build a `ThinkingQA` class

Wrap `compare_thinking()` in a class `ThinkingQA` with:
- `__init__(self, model, budget)` — stores config
- `query(self, prompt)` — calls the API, returns `{plain_answer, thought_answer, overhead_tokens}`
- `batch_query(self, prompts)` — runs multiple prompts, returns a list of results

Use it to run all three `PUZZLES` in one `batch_query()` call.

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 1: Add a new puzzle
# ============================================================

monty_hall = {
    "label": "Monty Hall",
    "prompt": (
        "You're on a game show with 3 doors. Behind one is a car; "
        "behind the others are goats. You pick door 1. The host (who knows "
        "what's behind each door) opens door 3, revealing a goat. "
        "Should you switch to door 2, or stay with door 1? "
        "What is the exact probability of winning if you switch vs stay?"
    ),
}

label  = monty_hall["label"]
prompt = monty_hall["prompt"]

print(f"[{label}]")
print(f"Q: {prompt}\n")

plain   = ask(prompt, thinking=False)
thought = ask(prompt, thinking=True)

print(f"Without thinking ({plain['output_tokens']} tokens):")
print(f"  {plain['answer'][:400]}")
print()
print(f"With thinking ({thought['output_tokens']} tokens, ~{thought['thinking_words']} words):")
print(f"  {thought['answer'][:400]}")

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 2: Budget sweep
# ============================================================

def budget_sweep(prompt: str, budgets: list[int]) -> list[dict]:
    """Run the same prompt at multiple thinking budgets and compare results."""
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    rows = []

    for budget in budgets:
        if budget == 0:
            r = client.messages.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1024,
            )
            answer = r.content[0].text.strip()
            thinking_words = 0
            tokens = r.usage.output_tokens
        else:
            r = client.messages.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=budget + 2048,
                thinking={"type": "enabled", "budget_tokens": budget},
            )
            thinking_text = ""
            answer = ""
            for block in r.content:
                if block.type == "thinking":
                    thinking_text = block.thinking
                elif block.type == "text":
                    answer = block.text.strip()
            thinking_words = len(thinking_text.split())
            tokens = r.usage.output_tokens

        rows.append({
            "budget": budget,
            "tokens": tokens,
            "thinking_words": thinking_words,
            "answer_snippet": answer[:150],
        })

    return rows


# Run sweep on the CRT puzzle
crt_prompt = PUZZLES[2]["prompt"]
sweep_results = budget_sweep(crt_prompt, budgets=[0, 1000, 4000, 8000])

print(f"Q: {crt_prompt}")
print()
print(f"{'Budget':>8} | {'Tokens':>8} | {'Think words':>12} | Answer snippet")
print("-" * 80)
for row in sweep_results:
    print(f"{row['budget']:>8,} | {row['tokens']:>8,} | {row['thinking_words']:>12,} | {row['answer_snippet']}")

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 3: ThinkingQA class
# ============================================================

class ThinkingQA:
    """Reusable wrapper for extended thinking comparisons."""

    def __init__(self, model: str = MODEL, budget: int = THINKING_BUDGET):
        self.model = model
        self.budget = budget
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    def query(self, prompt: str) -> dict:
        """Run prompt with and without thinking. Returns structured result."""
        # Without thinking
        r_plain = self.client.messages.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
        )
        plain_answer = r_plain.content[0].text.strip()
        plain_tokens = r_plain.usage.output_tokens

        # With thinking
        r_thought = self.client.messages.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=self.budget + 2048,
            thinking={"type": "enabled", "budget_tokens": self.budget},
        )
        thinking_text = ""
        thought_answer = ""
        for block in r_thought.content:
            if block.type == "thinking":
                thinking_text = block.thinking
            elif block.type == "text":
                thought_answer = block.text.strip()
        thought_tokens = r_thought.usage.output_tokens

        return {
            "prompt": prompt,
            "plain_answer": plain_answer,
            "thought_answer": thought_answer,
            "overhead_tokens": thought_tokens - plain_tokens,
            "thinking_words": len(thinking_text.split()),
        }

    def batch_query(self, prompts: list[str]) -> list[dict]:
        """Run multiple prompts sequentially, return list of results."""
        return [self.query(p) for p in prompts]


# Use it on all three puzzles
qa = ThinkingQA(budget=5000)
batch_results = qa.batch_query([p["prompt"] for p in PUZZLES])

print("ThinkingQA batch results:")
print()
for puzzle, result in zip(PUZZLES, batch_results):
    print(f"[{puzzle['label']}]")
    print(f"  Plain:   {result['plain_answer'][:150]}")
    print(f"  Thought: {result['thought_answer'][:150]}")
    print(f"  Overhead: +{result['overhead_tokens']} tokens | ~{result['thinking_words']} thinking words")
    print()

---
Workshop complete. Next: **example 97** — explore the next technique in the series.

### What you built today
- Direct `anthropic` SDK calls with `thinking={'type':'enabled','budget_tokens':N}`
- A two-block response parser handling `ThinkingBlock` and `TextBlock`
- A three-puzzle accuracy benchmark (modular math, wordplay, CRT)
- A `budget_sweep()` utility for quality-vs-cost analysis
- A reusable `ThinkingQA` class for batch evaluations

### Recommended next steps
- Read the [Anthropic Extended Thinking docs](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- Try combining extended thinking with [tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- Build an eval harness that scores accuracy across a larger benchmark dataset
- Experiment with streaming extended thinking responses